<a href="https://colab.research.google.com/github/JethroTababa/BeehealthBackendAPi/blob/main/Copy_of_Beewatch_hive_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart librosa numpy torch transformers accelerate

In [ ]:
import os
import shutil
import threading
import time

from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware

from transformers import pipeline

import uvicorn
from pyngrok import ngrok

In [ ]:
app = FastAPI(
    title="BeeWatch Pro Backend",
    description="BeeWatch Pro AI Hive Acoustic Classification API",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("✅ FastAPI app created")

✅ FastAPI app created


In [ ]:
MODEL_REPO = "troyskie/Beewatch-hive-classifier"

pipe = pipeline(
    "audio-classification",
    model=MODEL_REPO
)

print("===================================")
print("🐝 BeeWatch AST Model Loaded")
print("===================================")
print("Model:", MODEL_REPO)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

🐝 BeeWatch AST Model Loaded
Model: troyskie/Beewatch-hive-classifier


In [ ]:
@app.get("/")
def home():
    return {
        "status": "online",
        "message": "BeeWatch Backend Running",
        "model": MODEL_REPO
    }


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model_loaded": True,
        "model": MODEL_REPO
    }

In [ ]:
@app.post("/classify")
async def classify(audio_file: UploadFile = File(...)):

    # Temporary location for uploaded audio
    temp_path = f"/tmp/{audio_file.filename}"

    # Save uploaded WAV
    with open(temp_path, "wb") as buffer:
        shutil.copyfileobj(audio_file.file, buffer)

    print("\n===================================")
    print("🐝 BeeWatch Audio Received")
    print("===================================")
    print("Filename:", audio_file.filename)

    # Run your AST model
    results = pipe(temp_path)

    print("🤖 AST Results:")
    print(results)

    # Get highest probability
    prediction = max(
        results,
        key=lambda x: x["score"]
    )

    label = prediction["label"]
    confidence = prediction["score"]

    print("🐝 Prediction:", label)
    print("📊 Confidence:", confidence)

    # Remove temporary file
    if os.path.exists(temp_path):
        os.remove(temp_path)

    # Send result back to website
    return {
        "label": label,
        "confidence": round(float(confidence) * 100, 2),

        "all_predictions": [
            {
                "label": result["label"],
                "score": round(float(result["score"]), 4)
            }
            for result in results
        ]
    }

In [ ]:
!fuser -k 8000/tcp

In [ ]:
def run_uvicorn():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


server_thread = threading.Thread(
    target=run_uvicorn,
    daemon=True
)

server_thread.start()

time.sleep(3)

print("===================================")
print("🚀 FastAPI Server Started")
print("===================================")
print("Local server: http://127.0.0.1:8000")

INFO:     Started server process [4408]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 FastAPI Server Started
Local server: http://127.0.0.1:8000


In [ ]:
NGROK_AUTH_TOKEN = "3G9bB2sRm8FfUh4HfroyAy6rE4L_4v1mQSdKr9STv3srxHfSh"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)

print("======================================")
print("🐝 BeeWatch Pro Backend")
print("======================================")

print("\nBackend URL:")
print(public_url.public_url)

print("\nSwagger:")
print(public_url.public_url + "/docs")

print("\nClassify:")
print(public_url.public_url + "/classify")

🐝 BeeWatch Pro Backend

Backend URL:
https://droop-surcharge-getup.ngrok-free.dev

Swagger:
https://droop-surcharge-getup.ngrok-free.dev/docs

Classify:
https://droop-surcharge-getup.ngrok-free.dev/classify
